In [ ]:
print("Setup ready")

: 

In [ ]:
%pwd


In [ ]:
import os
os.chdir("../")

In [ ]:
%pwd

In [ ]:
from langchain.document_loaders import PyPDFLoader, DirectoryLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

In [ ]:
def load_pdf_files(data):
    loader = DirectoryLoader(
        data,
        glob="**/*.pdf",
        loader_cls=PyPDFLoader
        )
    documents = loader.load()
    return documents

In [ ]:
extracted_data = load_pdf_files("data")

In [ ]:
print(len(extracted_data))

In [ ]:
from typing import List
from langchain.schema import Document

def filter_to_minimal_docs(docs: List[Document]) -> List[Document]:
    minimal_docs :List[Document] = []
    for doc in docs:
        src = doc.metadata.get("source")
        minimal_docs.append(
            Document(
                page_content=doc.page_content,
                metadata={"source": src}
            )
        )
    return minimal_docs

In [ ]:
mininmal_docs = filter_to_minimal_docs(extracted_data)

In [ ]:
mininmal_docs

In [ ]:
#split the documents into chunks

def text_splitter(mininmal_docs):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=50,
        chunk_overlap=10,
        
    )
    texts_chunks = text_splitter.split_documents(mininmal_docs)
    return texts_chunks

In [ ]:
text_chunks = text_splitter(mininmal_docs)  
print(f"Number of text chunks: {len(text_chunks)}")

In [ ]:
text_chunks

In [ ]:
from langchain.embeddings import HuggingFaceEmbeddings
def download_embeddings():
    model_name = "sentence-transformers/all-MiniLM-L6-v2"
    embeddings = HuggingFaceEmbeddings(
        model_name=model_name
        ) 
    return embeddings

embeddings = download_embeddings()

In [ ]:
embeddings

In [ ]:
vector = embeddings.embed_query("Hello world")
print(vector)

In [ ]:
print("vector length", len(vector))

In [ ]:
from dotenv import load_dotenv
import os
load_dotenv()

In [ ]:
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")

os.environ["PINECONE_API_KEY"] = PINECONE_API_KEY
os.environ["OPENROUTER_API_KEY"] = OPENROUTER_API_KEY

In [ ]:
from pinecone import Pinecone
pinecone_api_key = PINECONE_API_KEY

pc = Pinecone(api_key = pinecone_api_key)

In [ ]:
pc

In [ ]:
from pinecone import ServerlessSpec
index_name = "aryan-chatbot"

if not pc.has_index(index_name):
    pc.create_index(
        name=index_name,
        dimension=384,
        metric="cosine",
       spec=ServerlessSpec(cloud ="aws", region="us-east-1")
    )
    
    index = pc.Index(index_name)

In [ ]:
import sys
print(sys.executable)


In [ ]:
!{sys.executable} -m pip uninstall pinecone -y
!{sys.executable} -m pip install pinecone==5.4.0

In [ ]:
import pinecone
print(pinecone.__version__)

In [ ]:
from langchain_pinecone import PineconeVectorStore

docsearch =  PineconeVectorStore.from_documents(
    documents = text_chunks,
    embedding=embeddings,       
    Index_name=index_name,
    
)